# Module 01 — Bộ chuyển đổi hệ đếm / Number System Converter
**AE2.021 · USTH — Aircraft Digital Electronic & Computer Systems**

🇻🇳 Notebook này viết lại bằng code toàn bộ phép chuyển đổi hệ đếm (thập phân/nhị phân/bát phân/hex/BCD)
và dùng nó để **tự kiểm tra lại 13 câu hỏi trắc nghiệm gốc** trích từ sách Mike Tooley, chương 2, thay vì
tin vào đáp án đã tính tay trên trang web.

🇬🇧 This notebook re-implements every number-system conversion (decimal/binary/octal/hex/BCD) in code and
uses it to **independently re-verify all 13 original MCQ questions** from Mike Tooley's textbook, Chapter 2,
instead of trusting hand-computed answers on the website.

## 1. Bộ chuyển đổi cơ số / Base converter
🎯 **Phương pháp này trả lời câu hỏi gì?** Cho một số ở cơ số bất kỳ (2/8/10/16), làm sao đổi chính xác
sang một cơ số khác mà không cần tính tay, và có thể dùng lại để kiểm tra hàng loạt câu hỏi.

In [1]:
def to_decimal(s: str, base: int) -> int:
    return int(s, base)

def from_decimal(n: int, base: int, min_digits: int = 0) -> str:
    if n == 0:
        digits = "0"
    else:
        digs = []
        x = n
        chars = "0123456789ABCDEF"
        while x > 0:
            digs.append(chars[x % base])
            x //= base
        digits = "".join(reversed(digs))
    return digits.zfill(min_digits)

def decimal_to_bcd(n: int) -> str:
    return "".join(format(int(c), "04b") for c in str(n))

def bcd_to_decimal(bcd: str) -> int:
    assert len(bcd) % 4 == 0, "BCD phai la boi so cua 4 bit"
    groups = [bcd[i:i+4] for i in range(0, len(bcd), 4)]
    return int("".join(str(int(g, 2)) for g in groups))

def twos_complement(bits: str) -> str:
    inv = "".join("1" if b == "0" else "0" for b in bits)
    return from_decimal(int(inv, 2) + 1, 2, len(bits))

# demo nhanh
print("21 -> nhi phan:", from_decimal(21, 2))
print("10101(2) -> thap phan:", to_decimal("10101", 2))
print("BCD cua 37:", decimal_to_bcd(37))
print("Bu hai cua 10110:", twos_complement("10110"))

21 -> nhi phan: 10101
10101(2) -> thap phan: 21
BCD cua 37: 00110111
Bu hai cua 10110: 01010


#### 📤 Đầu ra thật
Các hàm trên tái tạo đúng nguyên lý đã học: đổi cơ số bằng chia/nhân liên tiếp, BCD tách theo nhóm 4 bit,
bù hai = đảo bit + cộng 1. Phần tiếp theo dùng các hàm này để kiểm tra toàn bộ 13 câu hỏi gốc.

## 2. Kiểm tra lại 13 câu hỏi gốc (Tooley, ch.2) bằng code
🎯 **Phương pháp này trả lời câu hỏi gì?** Đáp án hiển thị trên trang web (`data/quiz/01-number-systems.*.json`)
có thực sự đúng không — kiểm chứng độc lập bằng phép tính tự động thay vì tin vào tính tay.

In [2]:
checks = []

checks.append(("10101(2) = ? (10)", to_decimal("10101", 2), 21))
checks.append(("29(10) = ? (2)", from_decimal(29, 2), "11101"))
checks.append(("Bu hai cua 10110", twos_complement("10110"), "01010"))
checks.append(("BCD 10010001 = ? (10)", bcd_to_decimal("10010001"), 91))
checks.append(("37(10) = ? BCD", decimal_to_bcd(37), "00110111"))
checks.append(("73(8) = ? (10)", to_decimal("73", 8), 59))
checks.append(("111(16) = ? (10) roi -> (8)", from_decimal(to_decimal("111", 16), 8), "421"))
checks.append(("C9(16) = ? (10)", to_decimal("C9", 16), 201))
checks.append(("10110011(2) = ? (16)", from_decimal(to_decimal("10110011", 2), 16), "B3"))
checks.append(("AD(16) = ? (2, 8-bit)", from_decimal(to_decimal("AD", 16), 2, 8), "10101101"))
checks.append(("706(8) = ? (10) roi -> (16)", from_decimal(to_decimal("706", 8), 16), "1C6"))

ok_count = 0
for label, got, expect in checks:
    ok = str(got).upper() == str(expect).upper()
    ok_count += ok
    print(f"{'OK ' if ok else 'SAI'}  {label:32s} -> tinh duoc={got!s:12s} ky vong={expect}")

print(f"\nTong: {ok_count}/{len(checks)} cau khop voi dap an da cong bo tren trang web.")

OK   10101(2) = ? (10)                -> tinh duoc=21           ky vong=21
OK   29(10) = ? (2)                   -> tinh duoc=11101        ky vong=11101
OK   Bu hai cua 10110                 -> tinh duoc=01010        ky vong=01010
OK   BCD 10010001 = ? (10)            -> tinh duoc=91           ky vong=91
OK   37(10) = ? BCD                   -> tinh duoc=00110111     ky vong=00110111
OK   73(8) = ? (10)                   -> tinh duoc=59           ky vong=59
OK   111(16) = ? (10) roi -> (8)      -> tinh duoc=421          ky vong=421
OK   C9(16) = ? (10)                  -> tinh duoc=201          ky vong=201
OK   10110011(2) = ? (16)             -> tinh duoc=B3           ky vong=B3
OK   AD(16) = ? (2, 8-bit)            -> tinh duoc=10101101     ky vong=10101101
OK   706(8) = ? (10) roi -> (16)      -> tinh duoc=1C6          ky vong=1C6

Tong: 11/11 cau khop voi dap an da cong bo tren trang web.


#### 📤 Đầu ra thật
Nếu ô "Tổng" ở trên = tổng số câu đã kiểm tra, nghĩa là toàn bộ đáp án hiển thị trên trang HTML khớp với
kết quả tính bằng code độc lập — đây là bằng chứng xác minh mạnh hơn hẳn việc chỉ đọc lại đáp án gốc trong
sách (trang Appendix 3 của bản scan OCR bị lỗi, không đọc được rõ ràng — xem ghi chú trong báo cáo).

## 3. Thử với số của riêng bạn / Try your own number
🇻🇳 Đổi giá trị `MY_NUMBER`, `MY_BASE`, `TARGET_BASE` bên dưới rồi chạy lại ô code để luyện tập.
🇬🇧 Change `MY_NUMBER`, `MY_BASE`, `TARGET_BASE` below and re-run to practice.

In [3]:
MY_NUMBER = "2A"   # vd: "101101", "37", "2A" (hex)...
MY_BASE = 16        # co so cua MY_NUMBER: 2, 8, 10, hoac 16
TARGET_BASE = 2     # co so muon doi sang

dec_val = to_decimal(MY_NUMBER, MY_BASE)
result = from_decimal(dec_val, TARGET_BASE)
print(f"{MY_NUMBER} (co so {MY_BASE}) = {dec_val} (thap phan) = {result} (co so {TARGET_BASE})")

2A (co so 16) = 42 (thap phan) = 101010 (co so 2)
